# CMap発現シグネチャから estradiol-like factor を探す

このノートブックでは、repo内に同梱した `ref_cmap.csv` を使い、化合物処理後の遺伝子発現シグネチャを解析します。

学生実習では、一括パイプラインではなく **1ステップ1セル** で進めます。

1. セットアップ
2. データサイズとheadの確認
3. 化合物間相関行列の可視化
4. DBSCANによるクラスタリング
5. PCA初期抽出 + varimax回転による潜在因子抽出
6. estradiol-high factor の上位・下位 sample の確認
7. 因子数の妥当性の確認

目的は Python を書くことではなく、**発現シグネチャの背後にある潜在軸をどう解釈するか**を体験することです。


## 0. 最初に

学生の方は、まず **[ドライブにコピー]** を押してください。
その後、このノートブックを上から順に実行してください。GitHubアカウントは不要です。


In [ ]:
#@title 1. セットアップ { display-mode: "form" }
# 通常は github のままで実行します。zipを直接試す場合は upload_zip に変更します。
install_mode = "github"  #@param ["github", "upload_zip", "skip"]

import sys
import subprocess
import matplotlib.pyplot as plt

if install_mode == "github":
    url = "https://github.com/mizuno-group/latent-demo/archive/refs/tags/v0.2.1.zip"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])
elif install_mode == "upload_zip":
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise RuntimeError("zipファイルをアップロードしてください。")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", zip_names[0]])

from latent_demo.cmap import (
    prepare_cmap_data,
    compute_compound_correlation,
    cluster_compounds_dbscan,
    fit_cmap_varimax_factors,
    compare_cmap_factor_numbers,
    plot_compound_correlation_heatmap,
    plot_dbscan_pca_scatter,
    plot_ranked_factor_scores,
    plot_cmap_factor_number_summary,
    ESTROGEN_TERMS,
    ANTI_ESTROGEN_TERMS,
    match_samples,
    summarize_terms,
)


## 1. 実習設定

まずはデフォルト値のまま実行してください。`n_components=40` は、estradiol-like な因子が比較的見えやすい設定です。


In [ ]:
#@title 2. パラメータを選ぶ { display-mode: "form" }
n_top_genes = 3000  #@param {type:"slider", min:500, max:6000, step:500}
n_components = 40  #@param {type:"slider", min:5, max:80, step:5}
target_sample = "estradiol"  #@param {type:"string"}
top_n = 20  #@param {type:"slider", min:5, max:40, step:5}
dbscan_eps = 0.6  #@param {type:"slider", min:0.3, max:0.9, step:0.05}
dbscan_min_samples = 3  #@param {type:"slider", min:2, max:8, step:1}


## 2. データを読み込む

repo内に同梱された `ref_cmap.csv` を読み込みます。行は遺伝子、列は化合物/sampleです。


In [ ]:
cmap = prepare_cmap_data(n_top_genes=n_top_genes)

print("genes used   :", cmap.gene_by_compound.shape[0])
print("compounds    :", cmap.gene_by_compound.shape[1])
print("target exists:", target_sample in cmap.compound_by_gene.index)


## 3. データサイズと head を見る

まずデータの向きを確認します。この段階では、まだモデリングはしていません。


In [ ]:
display(cmap.gene_by_compound.iloc[:5, :8])

print("compound x gene matrix:", cmap.compound_by_gene.shape)
display(cmap.compound_by_gene.iloc[:5, :8])


**問い**：このデータでは、何が sample で、何が feature でしょうか？


## 4. 化合物間相関行列を作る

化合物ごとの発現シグネチャが似ていれば、化合物間相関が高くなります。


In [ ]:
correlation = compute_compound_correlation(cmap.compound_by_gene)

display(correlation.iloc[:8, :8].round(3))
plot_compound_correlation_heatmap(
    correlation,
    title="Compound-compound correlation from expression signatures",
)
plt.show()


**問い**：相関が高い化合物は、何が似ていると解釈できるでしょうか？


## 5. DBSCANで化合物をクラスタリングする

相関距離 `1 - correlation` を使い、近い化合物の塊をDBSCANでざっくり見ます。


In [ ]:
clusters = cluster_compounds_dbscan(
    correlation,
    eps=dbscan_eps,
    min_samples=dbscan_min_samples,
)

cluster_summary = (
    clusters["cluster"].value_counts().sort_index()
    .rename_axis("cluster")
    .to_frame("n_samples")
)
display(cluster_summary)

plot_dbscan_pca_scatter(
    cmap.compound_by_gene,
    clusters,
    highlight_terms=["estradiol", "estrone", "estriol", "tamoxifen", "raloxifene", "fulvestrant", "clomifene"],
    title=f"DBSCAN on correlation distance (eps={dbscan_eps}, min_samples={dbscan_min_samples})",
)
plt.show()


In [ ]:
estrogen_hits = match_samples(cmap.compound_by_gene.index, ESTROGEN_TERMS)
anti_hits = match_samples(cmap.compound_by_gene.index, ANTI_ESTROGEN_TERMS)

key_clusters = clusters.loc[estrogen_hits + anti_hits].copy()
key_clusters["category"] = ["estrogen-like"] * len(estrogen_hits) + ["anti-estrogen"] * len(anti_hits)
display(key_clusters)


**問い**：estrogen-like 化合物と anti-estrogen 化合物は同じ塊に入るでしょうか？それとも分かれるでしょうか？


## 6. PCA初期抽出 + varimax回転で潜在因子を作る

Colabでの速度を優先し、ここでは通常の最尤FAではなく、PCAで低次元空間を作った後に varimax 回転します。
因子の符号は任意なので、後で `estradiol` が正になるようにそろえます。


In [ ]:
factor_result = fit_cmap_varimax_factors(
    cmap.compound_by_gene,
    n_components=n_components,
    target_sample=target_sample,
    n_top_genes=n_top_genes,
    random_state=0,
    select_mode="max_abs",
)

print("selected factor:", factor_result.selected_factor)
print(f"{target_sample} score:", round(factor_result.target_score, 3))
display(factor_result.scores.iloc[:5, :8].round(3))


**問い**：各化合物は、いくつの潜在軸上のスコアで表されているでしょうか？


## 7. estradiol-high factorの上位・下位を確認する

選ばれた因子のスコアで化合物をソートします。上位に estrogen-like、下位に anti-estrogen が来るかを確認します。


In [ ]:
top_samples = factor_result.ranked_scores.head(top_n)
bottom_samples = factor_result.ranked_scores.tail(top_n).sort_values("factor_score", ascending=True)

print(f"Top {top_n}: high side of {factor_result.selected_factor}")
display(top_samples.round(3))

print(f"Bottom {top_n}: opposite side of {factor_result.selected_factor}")
display(bottom_samples.round(3))

plot_ranked_factor_scores(
    factor_result.ranked_scores,
    top_n=top_n,
    title=f"Samples ranked by {factor_result.selected_factor} score (target={target_sample})",
)
plt.show()


In [ ]:
key_scores = summarize_terms(factor_result.ranked_scores)
key_scores = key_scores.sort_values("factor_score", ascending=False)
display(key_scores.round(3))


**問い**：Top側に estrogen-like な化合物、Bottom側に anti-estrogen が来るでしょうか？

**補足**：この結果は、因子数や遺伝子選択に依存します。`n_components=10` と `n_components=40` を比較してみてください。


## 8. 発展：因子数の妥当性を眺める

因子数を増やすと再構成誤差は下がりやすくなります。一方、テスト対数尤度や解釈性を見ると、単純に多ければよいとは限りません。


In [ ]:
#@title 3. 因子数を比較する { display-mode: "form" }
component_grid_text = "5,10,20,40,60"  #@param {type:"string"}
component_grid = tuple(int(x.strip()) for x in component_grid_text.split(",") if x.strip())

factor_summary = compare_cmap_factor_numbers(
    cmap.compound_by_gene,
    component_grid=component_grid,
    random_state=0,
)
display(factor_summary.round(4))

plot_cmap_factor_number_summary(
    factor_summary,
    title="PPCA-like held-out likelihood by component number",
)
plt.show()


**問い**：尤度や再構成誤差でよさそうな因子数と、estradiol-like factor が見えやすい因子数は一致するでしょうか？


## 9. まとめ

- 遺伝子発現シグネチャを使うと、化合物同士の近さをデータ駆動的に見ることができる。
- DBSCAN は、似ている化合物の塊をざっくり可視化するための道具になる。
- PCA初期抽出 + varimax回転により、特定の化合物が強く反映される探索的な潜在軸を見つけられる。
- 因子の符号は任意であり、ここでは estradiol が正になるように向きをそろえている。
- 因子数の選択は、尤度、再構成誤差、解釈性を合わせて考える必要がある。
